**structured outputs** have replaced **message prefill** which are now deprecated

 You give the API a JSON schema through output_config.format, and it constrains the response to match. That's stricter than prefill ever was.

In [3]:
# Init
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv(override=True)

api_key = os.getenv("API_KEY")
base_url = os.getenv("BASE_URL")
model = os.getenv("MODEL")

client = Anthropic(api_key=api_key, base_url=base_url)

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


In [4]:
import json

schema = {
    "type": "object",
    "properties": {
        "Name": {"type": "string"},
        "Description": {"type": "string"},
        "State": {"type": "string", "enum": ["ENABLED", "DISABLED"]},
        "EventBusName": {"type": "string"},
        "EventPattern": {
            "type": "object",
            "properties": {
                "source": {"type": "array", "items": {"type": "string"}},
                "detail-type": {"type": "array", "items": {"type": "string"}},
            },
            "required": ["source", "detail-type"],
            "additionalProperties": False,
        },
        "Targets": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "Id": {"type": "string"},
                    "Arn": {"type": "string"},
                },
                "required": ["Id", "Arn"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["Name", "State", "EventPattern", "Targets"],
    "additionalProperties": False,
}

messages = []
add_user_message(messages, "generate an example AWS::Events::Rule event bridge rule as json")

message = client.messages.create(
    model=model,
    max_tokens=5000,
    messages=messages,   # must end with a user message
    output_config={"format": {"type": "json_schema", "schema": schema}},
)
text = "".join(b.text for b in message.content if b.type == "text")
data = json.loads(text)
data

{'Name': 'ec2-instance-state-change-rule',
 'Description': 'Triggers a Lambda function when an EC2 instance changes state',
 'State': 'ENABLED',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification']},
 'Targets': [{'Id': 'ProcessEC2StateChangeLambda',
   'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:process-ec2-state-change'}]}